# Week 10 Live Session: Sequences, Attention & Transformers

## What you'll do today

1. **Recap** Week 9 (transfer learning for vision) — same idea, different domain today
2. **Brief funeral for RNN/LSTM** — what came before transformers
3. **Attention deep dive** — Q/K/V intuition, no math derivations
4. **Hugging Face fundamentals** — `pipeline`, `AutoTokenizer`, `from_pretrained`
5. **Fine-tune DistilBERT** on text classification

## Continuity from Week 9
Last week: pretrained CNN backbones for vision. Today: pretrained transformers for text. **Same `from_pretrained` pattern, different framework, different architecture.**

## Foreshadow Week 11
When you scale a decoder transformer to billions of parameters and train it on the entire internet — that's GPT. Next week.

## Section 0: Setup

**You should have a free Hugging Face account from Week 9 post-class.** If not, create one at https://huggingface.co — takes 2 minutes.

In [ ]:
# Install if not already (usually pre-installed on the course platform)
# !pip install transformers datasets evaluate 'accelerate>=1.1.0'

import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

set_seed(42)

# This notebook runs on GPU, Apple Silicon, or plain CPU — no code changes needed.
if torch.cuda.is_available():
    print(f'GPU (CUDA): {torch.cuda.get_device_name(0)} — fine-tuning ~30-60 sec')
elif torch.backends.mps.is_available():
    print('Apple Silicon (MPS) — fine-tuning ~3-5 min')
else:
    print('CPU only — fine-tuning ~5-10 min (works fine, just slower)')

## Section 1: Hugging Face `pipeline` — the easiest API in ML

Before we go deep on architecture, let's see what Hugging Face makes easy. **3 lines of code → working sentiment classifier.**

In [ ]:
sentiment = pipeline('sentiment-analysis')

# Try it on some examples
examples = [
    'I absolutely love this product!',
    'This is the worst experience I have ever had.',
    'The package arrived on time, exactly as described.',
    "It's fine, I guess.",
]

for text in examples:
    result = sentiment(text)[0]
    print(f'{result["label"]:8s} ({result["score"]:.3f}) | {text}')

**That just worked.** No data preparation, no training, no model architecture details. Hugging Face downloaded a pretrained sentiment model and wrapped it in a `pipeline` you can call.

**But — what's actually happening underneath?** That's what we'll spend most of class on.

In [ ]:
# Summarization — an ENCODER-DECODER (seq2seq) task.
# Heads up: transformers v5 removed the 'summarization' pipeline shortcut, so we drive the
# model directly with AutoModelForSeq2SeqLM + .generate(). Bonus: this lets us SEE the
# encoder-decoder family in action (the same .generate() call you'll use for LLMs in Week 11).
from transformers import AutoModelForSeq2SeqLM

sum_name = 'google-t5/t5-small'          # T5 is "text-to-text": ONE model does many tasks via a task prefix
sum_tok = AutoTokenizer.from_pretrained(sum_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_name)

long_text = '''
In 2017, a paper titled "Attention Is All You Need" introduced the Transformer architecture.
It replaced recurrent neural networks for sequence modeling tasks. The key innovation was the
attention mechanism, which allows the model to consider all positions in the sequence in
parallel rather than processing them sequentially. This made training dramatically faster and
enabled scaling to much larger models. Within 5 years, transformers had become the dominant
architecture not only for language but also for vision, audio, code, and biological sequences.
'''

# The 'summarize: ' prefix tells T5 which task to perform.
inputs = sum_tok('summarize: ' + long_text, return_tensors='pt', truncation=True, max_length=512)
summary_ids = sum_model.generate(**inputs, max_new_tokens=50, num_beams=4)
print('Summary:')
print(sum_tok.decode(summary_ids[0], skip_special_tokens=True))

In [ ]:
# Zero-shot classification — classify into ANY classes you specify (no training!)
zero_shot = pipeline('zero-shot-classification')

text = 'I just bought tickets to see the Yankees play next weekend.'
candidate_labels = ['sports', 'politics', 'food', 'technology', 'travel']

result = zero_shot(text, candidate_labels)
print(f'Text: {text}\n')
for label, score in zip(result['labels'], result['scores']):
    print(f'  {label:12s}: {score:.3f}')

**This works without ANY training on your data.** Zero-shot classification leverages the model's general language understanding to figure out which label best fits.

Now: how does this work? Time for attention.

## Section 2: Attention Mechanism (the heart of the lesson)

**The intuition: attention is a learnable lookup.**

Imagine the network is processing the sentence: `The cat sat on the [???]`

To predict the next word, the network needs to look back at the earlier words. But which ones matter most?
- `The` — function word, low signal
- `cat` — high signal! cats sit on things
- `sat` — moderate signal, indicates the verb is done
- `on` — high signal! locative preposition
- `the` — function word, low signal

**Attention computes which earlier tokens are most relevant for each prediction step.** It does this with a Query, Key, Value mechanism.

### Q/K/V analogy
Think of it like a search engine inside the network:
- **Query (Q):** "What am I looking for?" Each token issues a query.
- **Key (K):** "What do I offer?" Every token offers a key.
- **Match score:** Compute Q·K for each pair (high = relevant)
- **Softmax:** Normalize match scores into a probability distribution
- **Value (V):** "Here's my actual content." Each token has a value
- **Output:** Weighted sum of Values, weighted by the match scores

**Same input embedding gets projected three different ways** (Q, K, V) — three different learned roles for the same word.

### Multi-head attention
We do this multiple times in parallel, with different learned projections. Each "head" learns to attend to different relationships (subject-verb, modifier-noun, etc.). Outputs are concatenated.

### Positional encoding
Transformers process all positions in parallel — they don't know order from architecture alone. **Positional encodings (sine/cosine waves) are added to token embeddings to inject position information.**

In [ ]:
# Visualize attention weights on a real sentence (using bertviz-style approach)
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModel.from_pretrained('distilbert-base-uncased', output_attentions=True)

sentence = 'The cat sat on the mat'
inputs = tokenizer(sentence, return_tensors='pt')
outputs = model(**inputs)

# attentions: tuple of length num_layers, each [batch, num_heads, seq_len, seq_len]
attention = outputs.attentions[5][0]  # last layer, first batch element
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Average across heads for visualization
attn_avg = attention.mean(dim=0).detach().numpy()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attn_avg, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_xlabel('Key (attended to)')
ax.set_ylabel('Query (attending from)')
ax.set_title('Attention weights (averaged across heads, last layer)')
plt.colorbar(im); plt.tight_layout(); plt.show()

print('\nReading the heatmap: each row shows what one token attends to.')
print('Bright spots = high attention. The diagonal pattern is partly self-attention.')

## Section 3: The Transformer Family Tree

Three families, each useful for different tasks:

| Family | Examples | Best for |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa, DistilBERT, DeBERTa | Understanding tasks: classification, NER, embedding |
| **Decoder-only** | GPT-2/3/4, Llama, Qwen, Phi | Generation tasks: text completion, dialogue |
| **Encoder-decoder** | T5, BART, mBART | Sequence-to-sequence: translation, summarization |

**Today we'll fine-tune an encoder-only model (DistilBERT) for classification.** Next week we'll use a decoder-only (Phi or Qwen) for generation.

## Section 4: Fine-Tune DistilBERT for Text Classification

Same `from_pretrained` pattern from Week 9, but for transformers.

**Task:** Classify news articles into 4 categories (World, Sports, Business, Sci/Tech) using AG News dataset.

**Why DistilBERT:** smaller and faster than BERT (40% fewer parameters, 60% faster), retains 97% of BERT's accuracy on most tasks. Perfect for fine-tuning demos.

In [ ]:
# Load AG News dataset (4 classes: World, Sports, Business, Sci/Tech)
ds = load_dataset('ag_news')

# Subset to 2K training samples for fast demo
train_ds = ds['train'].select(range(2000))
test_ds = ds['test'].select(range(500))

print(f'Training: {len(train_ds)} examples')
print(f'Test: {len(test_ds)} examples')
print(f'\nClasses: {train_ds.features["label"].names}')
print(f'\nExample:\n  Text: {train_ds[0]["text"][:150]}...')
print(f'  Label: {train_ds.features["label"].names[train_ds[0]["label"]]}')

In [ ]:
# CRITICAL: load tokenizer matching the model
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Show what tokenization looks like
example_text = train_ds[0]['text']
tokens = tokenizer(example_text, truncation=True, max_length=128)
print(f'Original text: {example_text[:100]}...')
print(f'\nToken IDs (first 20): {tokens["input_ids"][:20]}')
print(f'Decoded tokens: {tokenizer.convert_ids_to_tokens(tokens["input_ids"][:20])}')

**Notice:** the tokenizer breaks text into subword pieces. `[CLS]` is a special token at the start; `[SEP]` ends the sequence. The `##` prefix marks subword continuations. **Each model has its own tokenizer — always use the matching one.**

In [ ]:
# Tokenize the entire dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

train_tokenized = train_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

print(f'Tokenization complete')
print(f'Each example now has: {train_tokenized.column_names}')

In [ ]:
# Load the model with a fresh classification head for 4 classes
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# How many parameters?
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# Set up training (HF's Trainer API — analogous to model.fit)
training_args = TrainingArguments(
    output_dir='./distilbert_agnews',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,  # CRITICAL: low LR for fine-tuning (same lesson as Week 9)
    eval_strategy='epoch',
    logging_steps=50,
    save_strategy='no',
    report_to='none',
    disable_tqdm=True,   # plain-text progress: avoids a transformers v5 notebook
                         # progress-bar callback bug and the ipywidgets dependency
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = (predictions == labels).mean()
    return {'accuracy': accuracy}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

# Train! On GPU: ~30-60 sec. On CPU: 5-10 min.
trainer.train()

In [ ]:
# Evaluate
eval_results = trainer.evaluate()
print(f'\nFinal test accuracy: {eval_results["eval_accuracy"]:.4f}')

## Section 5: Use the Fine-Tuned Model

In [ ]:
# Predict on new examples
label_names = train_ds.features['label'].names

def predict(text, model, tokenizer):
    # Use the model's OWN device so inputs and weights always match —
    # works on CPU, CUDA, and Apple Silicon (MPS) without special-casing.
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
    pred_idx = probs.argmax()
    return label_names[pred_idx], probs[pred_idx]

test_examples = [
    'The Lakers won their game last night against the Celtics.',
    'NASA launched a new satellite to study deep space phenomena.',
    'Inflation rose 3.2% according to the latest economic report.',
    'The president gave a speech at the United Nations today.',
]

for text in test_examples:
    label, conf = predict(text, model, tokenizer)
    print(f'[{label:9s}] ({conf:.3f}) {text}')

## Section 6: Save the Model

In [ ]:
model.save_pretrained('./week10_distilbert_agnews')
tokenizer.save_pretrained('./week10_distilbert_agnews')
print('Saved model and tokenizer to ./week10_distilbert_agnews/')

# Reload (for inference later)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
loaded_model = AutoModelForSequenceClassification.from_pretrained('./week10_distilbert_agnews')
loaded_tokenizer = AutoTokenizer.from_pretrained('./week10_distilbert_agnews')
print('Reloaded successfully')

## Wrap-up

Today:
1. **Used HF `pipeline`** for ready-made tasks (sentiment, summary, zero-shot)
2. **Internalized the attention intuition** — Q/K/V learnable lookup
3. **Toured the three transformer families** (encoder-only, decoder-only, encoder-decoder)
4. **Fine-tuned DistilBERT** on AG News — same pretrained-model pattern as Week 9

## Looking ahead: Week 11

Encoder transformers (today) UNDERSTAND text. Decoder transformers (next week) GENERATE text. Scale up a decoder transformer to billions of parameters, train it on the entire internet, fine-tune with human feedback — that's ChatGPT.

Next week: GENERATIVE AI. LLMs and diffusion models. Course finale before the Copilot Workshop in Week 12.

## Post-class
Open `week10_pair_programming.ipynb` to explore Hugging Face pipelines, then `week10_postclass_exercise.ipynb` to build a HF pipeline for a personal text task.